# Eval Pivot 5 — Qwen3.5-0.8B: baseline vs fine-tuned (Colab)

Notebook ini **memanggil** `eval.py`, `merge_adapter.py`, `export_gguf.py` (semua skrip `.py`
ada di folder proyek pada Google Drive). Jalankan sel berurutan.

## Prasyarat di folder Drive proyek (PROJECT_DIR):
- `eval.py`, `merge_adapter.py`, `export_gguf.py`, `chat_format.py`  ← upload versi terbaru
- `Data/processed_id/test.jsonl`  (2998 sampel)
- `outputs/checkpoints/qwen35-0.8b-train/`  (adapter + checkpoint-5628)

> Runtime: **GPU** (Runtime → Change runtime type → T4/L4). Output eval tersimpan ke
> `results/` di Drive (persisten).

In [ ]:
# 1) Mount Drive + masuk ke folder proyek
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/Aries/Fine-Tune SLM for Medical Chatbot'  # sesuaikan bila beda
%cd "$PROJECT_DIR"
!ls eval.py merge_adapter.py export_gguf.py chat_format.py Data/processed_id/test.jsonl outputs/checkpoints/qwen35-0.8b-train

In [ ]:
# 2) Dependensi (unsloth = loader model; bert-score = metrik semantik)
!pip install -q unsloth unsloth_zoo bert-score
# untuk eval GGUF (langkah 6) — boleh dijalankan nanti:
# !pip install -q llama-cpp-python

In [ ]:
# 3) Eval BASELINE (pre-trained, 16-bit) — full test set, + BERTScore
!python eval.py --model unsloth/Qwen3.5-0.8B \
                --label qwen08_baseline --n_eval 3000 --bertscore

In [ ]:
# 4) Eval FINE-TUNED 16-bit (adapter langsung, tanpa merge) — referensi quantization gap
!python eval.py --model outputs/checkpoints/qwen35-0.8b-train \
                --label qwen08_finetuned --n_eval 3000 --bertscore

In [ ]:
# 5) Merge adapter -> 16-bit (WAJIB sebelum GGUF). Hasil: outputs/merged/qwen35-0.8b-medical
!python merge_adapter.py \
    --adapter outputs/checkpoints/qwen35-0.8b-train \
    --out     outputs/merged/qwen35-0.8b-medical

In [ ]:
# 6) Export GGUF Q4_K_M (build llama.cpp otomatis). Hasil: outputs/gguf/...-Q4_K_M.gguf
!pip install -q llama-cpp-python
!python export_gguf.py --merged_dir outputs/merged/qwen35-0.8b-medical --verify

In [ ]:
# 7) Eval FINE-TUNED Q4_K_M (angka deployment jujur)
!python eval.py --gguf outputs/gguf/qwen35-0.8b-medical-Q4_K_M.gguf \
                --model outputs/merged/qwen35-0.8b-medical \
                --label qwen08_finetuned_q4 --loader gguf --n_eval 3000 --bertscore

In [ ]:
# 8) RINGKAS -> tabel delta (base vs fine-tuned) + quantization gap (16bit vs Q4)
#    + cek ROUGE-L >= 0.30. Semua dari results/*.json di Drive.
!python eval.py --summarize results